# Section 1 — Movie Recommendation System
**ML Case Study 122 · B.Tech CSE Sem V**

| Model | Role |
|---|---|
| **KNN** | Primary recommendation engine (similar movies) |
| **K-Means** | Movie segmentation |
| **Hierarchical** | Movie grouping + dendrogram |
| **Random Forest** | Predict whether a user will like a movie |

**Pipeline:** Data → Cleaning → EDA → Features → 4 Models → Analysis → Final KNN Recommender → Streamlit

# Section 2 — Problem Definition
| | |
|---|---|
| **Problem** | A media platform wants to use movie data and user ratings to give personalised movie suggestions |
| **Objective** | Use user–item interactions (ratings) to give **personalised** movie suggestions, and also similar-movie suggestions |
| **Input** | A user's rating history (personalised) or a movie title (similar movies); user history + movie features (Random Forest) |
| **Output** | Top-N movies for a user · Top-N similar movies · movie clusters · P(user likes movie) |
| **ML approach** | KNN (similarity) · K-Means & Hierarchical (unsupervised) · Random Forest (supervised) |

**How interaction data is used (justification):** (1) a user's *liked* movies (rating ≥ 4) become seeds for personalised KNN recommendations, (2) held-out liked movies measure recommendation quality, (3) user/movie rating history are Random Forest features.

**Target for Random Forest:** `like = 1` if `rating >= 4`, else `0` (derived from real ratings).

# Section 3 — Dataset & Documentation
- **File:** `movie_ratings_merged.csv` — one row per user rating, merged with movie metadata (user–item interaction data + content data).
- **Why this dataset:** it has *both* user–item ratings (personalisation) and movie content (genres, overview) → supports similarity, clustering and preference prediction.
- **Source:** `__________` *(write your dataset source / link here)*
- **Key columns:** `userId`, `movieId`, `rating` (interaction) · `title`, `genres`, `overview`, `release_year`, `vote_average`, `vote_count` (movie)
- **Limitations:** very sparse user–item matrix · popular movies dominate · no user demographics.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy.sparse import hstack
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, adjusted_rand_score, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

SEED = 42
np.random.seed(SEED)
os.makedirs("models", exist_ok=True)

df = pd.read_csv("movie_ratings_merged.csv")
need = ["userId", "movieId", "rating", "title", "genres", "overview", "release_year", "vote_average", "vote_count"]
assert all(c in df.columns for c in need), "Required column missing"

missing_pct = (df.isnull().mean() * 100).round(2)
print("Shape:", df.shape, "| Users:", df.userId.nunique(), "| Movies:", df.movieId.nunique())
print("\nData types:\n", df.dtypes)
print("\nMissing % :\n", missing_pct[missing_pct > 0])
print("\nDuplicate rows:", df.duplicated().sum(), "| Duplicate (user, movie):", df.duplicated(["userId", "movieId"]).sum())
df.head(3)

# Section 4 — Data Cleaning
Fix data types → remove duplicates → fill missing values (median / 0 / empty text) → clean text & genres → create `like` target.

In [ ]:
n_before = len(df)
for c in ["rating", "release_year", "vote_average", "vote_count"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")                       # correct data types
df = df.dropna(subset=["userId", "movieId", "rating", "title"])
df = df.drop_duplicates().drop_duplicates(["userId", "movieId"], keep="last").reset_index(drop=True)
df[["userId", "movieId"]] = df[["userId", "movieId"]].astype(int)

df["release_year"] = df.release_year.fillna(df.release_year.median()).astype(int)   # missing values
df["vote_average"] = df.vote_average.fillna(df.vote_average.median())
df["vote_count"] = df.vote_count.fillna(0)

df["genres"] = df.genres.fillna("Unknown").replace("(no genres listed)", "Unknown")
df["genres_clean"] = df.genres.str.replace("|", " ", regex=False).str.replace(r"[^A-Za-z ]", "", regex=True)  # Sci-Fi -> SciFi
df["overview"] = df.overview.fillna("").str.replace(r"[^A-Za-z0-9\s]", " ", regex=True)                       # text cleaning
df["like"] = (df.rating >= 4.0).astype(int)                                                                  # target

print(f"Rows: {n_before:,} -> {len(df):,}")
print(df["like"].value_counts(normalize=True).round(3))

# Section 5 — Exploratory Data Analysis
Rating distribution · movie popularity · top movies · genres · missing values · correlation.

In [ ]:
movie_stats = df.groupby("movieId").agg(n_ratings=("rating", "size"), mean_rating=("rating", "mean"),
                                        like_rate=("like", "mean")).reset_index()
movie_info = df.drop_duplicates("movieId").merge(movie_stats, on="movieId")
genre_counts = movie_info.genres_clean.str.split().explode().value_counts()
print(f"User-item matrix density: {len(df) / (df.userId.nunique() * df.movieId.nunique()):.2%}")

fig, ax = plt.subplots(2, 3, figsize=(17, 9))
df.rating.value_counts().sort_index().plot.bar(ax=ax[0, 0], color="teal", title="Rating distribution")
ax[0, 1].hist(movie_stats.n_ratings, bins=40, color="slateblue"); ax[0, 1].set(title="Ratings per movie (popularity)", yscale="log")
movie_info.nlargest(10, "n_ratings").set_index("title").n_ratings[::-1].plot.barh(ax=ax[0, 2], color="seagreen", title="Top 10 most-rated movies")
genre_counts.head(12)[::-1].plot.barh(ax=ax[1, 0], color="crimson", title="Genre distribution")
missing_pct[missing_pct > 0].plot.barh(ax=ax[1, 1], color="gray", title="Missing values in raw data (%)")
sns.heatmap(movie_info[["n_ratings", "mean_rating", "vote_average", "vote_count", "release_year"]].corr(),
            annot=True, fmt=".2f", cmap="coolwarm", ax=ax[1, 2]); ax[1, 2].set_title("Correlation")
plt.tight_layout(); plt.show()

# Section 6 — Feature Engineering
| Technique | Used for | Why |
|---|---|---|
| **TF-IDF** (genres + overview) | KNN, clustering | converts text/categories to numbers, down-weights common words |
| **L2 normalisation** | KNN | cosine similarity compares content, not text length |
| **SVD (100 dims)** | clustering | compact dense features for K-Means / Ward |
| **StandardScaler** | year, vote avg, vote count | same scale for distance-based clustering |
| **Genre flags (0/1)** | Random Forest | simple tree-friendly inputs (no scaling needed) |

In [ ]:
movies = movie_info[["movieId", "title", "genres", "genres_clean", "overview", "release_year", "vote_average",
                     "vote_count", "n_ratings", "mean_rating", "like_rate"]].reset_index(drop=True)
movies["title_key"] = movies.title.str.lower().str.strip()
movie_row = dict(zip(movies.movieId, movies.index))                       # movieId -> matrix row

# TF-IDF on genres and overview, combined and normalised
tfidf_g = TfidfVectorizer(token_pattern=r"\S+")
tfidf_t = TfidfVectorizer(stop_words="english", max_features=5000, min_df=2)
G = tfidf_g.fit_transform(movies.genres_clean)
T = tfidf_t.fit_transform(movies.overview)
X_content = normalize(hstack([G, T]).tocsr())

# Dense features for clustering: SVD of content + scaled numeric columns
Z = normalize(TruncatedSVD(n_components=min(100, X_content.shape[1] - 1), random_state=SEED).fit_transform(X_content))
num = StandardScaler().fit_transform(np.c_[movies.release_year, movies.vote_average, np.log1p(movies.vote_count)])
X_cluster = np.hstack([Z, 0.3 * num])

# Genre flags for Random Forest
top_genres = genre_counts.drop("Unknown", errors="ignore").head(12).index
for g in top_genres:
    movies["g_" + g] = movies.genres_clean.str.split().apply(lambda t: int(g in t))
genre_cols = ["g_" + g for g in top_genres]
print("Movies:", len(movies), "| Content matrix:", X_content.shape, "| Clustering matrix:", X_cluster.shape)

# Section 7 — Model 1: KNN (Primary Recommender)
- **Goal:** find the movies most similar to a chosen movie (retrieval, not classification).
- **How:** cosine distance on TF-IDF vectors; `similarity = 1 − distance`.
- **Personalised version:** for a user, take neighbours of *every movie they liked*, add up similarity scores, hide already-rated movies → user's Top-N. This is how interaction data drives suggestions.
- **Experiment:** genres only vs overview only vs genres + overview (final).
- **Evaluation:** hide 20% of each user's liked movies, recommend from the rest, measure **Precision@10, Recall@10, Hit-rate@10**; compared with *Random* and *Most-popular* baselines.

In [ ]:
# ---- Evaluation set: users with >= 6 liked movies; seeds = 80% of likes, held-out = 20% ----
rng = np.random.default_rng(SEED)
liked = df[df.like == 1].groupby("userId").movieId.apply(list)
liked = liked[liked.str.len() >= 6]
splits = []
for u in rng.choice(liked.index, min(300, len(liked)), replace=False):
    items = rng.permutation(liked[u]); n_test = max(1, int(0.2 * len(items)))
    splits.append((items[n_test:][:20], items[:n_test]))

def make_knn(X):
    knn = NearestNeighbors(metric="cosine", algorithm="brute").fit(X)
    def recs(seeds, k=10):                                              # personalised: neighbours of all seed movies
        rows = [movie_row[m] for m in seeds]
        dist, ind = knn.kneighbors(X[rows], n_neighbors=51)
        score = {}
        for d_row, i_row in zip(dist, ind):
            for d, i in zip(d_row, i_row):
                if i not in rows:
                    score[i] = score.get(i, 0) + 1 - d                  # sum similarity over all seed movies
        return set(movies.movieId.iloc[sorted(score, key=score.get, reverse=True)[:k]])
    return knn, recs

pop = movies.sort_values("n_ratings", ascending=False).movieId.tolist()
def pop_recs(seeds, k=10):
    s = set(seeds); return set([m for m in pop if m not in s][:k])
def rand_recs(seeds, k=10):
    return set(rng.choice(np.setdiff1d(movies.movieId, seeds), k, replace=False))

def evaluate(fn):
    p = r = h = 0
    for seeds, test in splits:
        hits = len(fn(seeds) & set(test)); p += hits / 10; r += hits / len(test); h += hits > 0
    return {"Precision@10": p / len(splits), "Recall@10": r / len(splits), "HitRate@10": h / len(splits)}

# ---- Experiments: which features work best for KNN? (final model = genres + overview) ----
FINAL = "KNN: genres + overview"
variants = {"KNN: genres only": normalize(G), "KNN: overview only": normalize(T), FINAL: X_content}
knns = {name: make_knn(X) for name, X in variants.items()}
knn_model, knn_recs = knns[FINAL]
res = {name: evaluate(f) for name, (_, f) in knns.items()}
res["Most popular"], res["Random"] = evaluate(pop_recs), evaluate(rand_recs)
knn_results = pd.DataFrame(res).T.round(4)
print("Users evaluated:", len(splits)); print(knn_results)
print("Movies with empty overview:", (movies.overview.str.strip() == "").sum(), "of", len(movies))

# Example: neighbours of the most-rated movie
q = int(movies.n_ratings.idxmax())
dist, ind = knn_model.kneighbors(X_content[q], n_neighbors=6)
print("\nSimilar to:", movies.title[q])
movies.iloc[ind[0][1:]][["title", "genres", "release_year"]].assign(similarity=(1 - dist[0][1:]).round(3))

**Result analysis:** precision/recall are small for all methods because only *held-out liked movies* count as correct. Compare (1) the three KNN variants → which features carry the signal, (2) KNN vs *Random* and *Most popular*. If *Most popular* is higher, that is a real finding: content-only KNN is strong for item-to-item similarity and cold-start, but popularity captures taste shared by most users. Empty-overview movies get weaker text similarity (limitation).

# Section 8 — Model 2: K-Means (Movie Segmentation)
- **K-Means** divides movies into **K predefined clusters** around centroids.
- **K selection:** Elbow (inertia) + Silhouette for K = 2…12; final K = best silhouette with **K ≥ 4** (K = 2 is too coarse).
- Cluster names come from the data (top genres/words), not hard-coded.

In [ ]:
def sil(X, labels):
    return silhouette_score(X, labels, sample_size=min(5000, len(X)), random_state=SEED)

ks = range(2, min(12, len(movies) - 1) + 1)
fits = {k: KMeans(k, n_init=10, random_state=SEED).fit(X_cluster) for k in ks}
scores = pd.DataFrame({"k": list(ks), "inertia": [fits[k].inertia_ for k in ks],
                       "silhouette": [sil(X_cluster, fits[k].labels_) for k in ks]})
best_k = int(scores[scores.k >= 4].sort_values("silhouette").k.iloc[-1])

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(scores.k, scores.inertia, "o-"); ax[0].set(title="Elbow Method", xlabel="K", ylabel="Inertia")
ax[1].plot(scores.k, scores.silhouette, "o-r"); ax[1].set(title="Silhouette vs K", xlabel="K")
for a in ax: a.axvline(best_k, ls="--", c="gray")
plt.tight_layout(); plt.show()

kmeans = fits[best_k]
movies["kmeans_cluster"] = kmeans.labels_
km_sil = sil(X_cluster, kmeans.labels_)
print(f"Selected K = {best_k} | inertia = {kmeans.inertia_:.1f} | silhouette = {km_sil:.4f}")

In [ ]:
gnames, tnames, nG = tfidf_g.get_feature_names_out(), tfidf_t.get_feature_names_out(), G.shape[1]

def describe(labels, rows):
    # Cluster profile from the average TF-IDF vector of its movies
    out = []
    for c in np.unique(labels):
        m = rows[labels == c]; cen = np.asarray(X_content[m].mean(axis=0)).ravel()
        g, t, s = gnames[cen[:nG].argsort()[::-1][:3]], tnames[cen[nG:].argsort()[::-1][:5]], movies.iloc[m]
        out.append({"cluster": c, "size": len(m), "label": " / ".join(g[:2]), "top_genres": ", ".join(g),
                    "top_words": ", ".join(t), "median_year": int(s.release_year.median()),
                    "avg_user_rating": round(s.mean_rating.mean(), 2)})
    return pd.DataFrame(out)

pca = PCA(2, random_state=SEED).fit(X_cluster)
def plot2d(rows, labels, title):
    xy = pca.transform(X_cluster[rows])
    plt.figure(figsize=(7, 5)); plt.scatter(xy[:, 0], xy[:, 1], c=labels, cmap="tab20", s=10)
    plt.colorbar(label="cluster"); plt.title(title); plt.xlabel("PCA 1"); plt.ylabel("PCA 2"); plt.show()

all_rows = np.arange(len(movies))
plot2d(all_rows, kmeans.labels_, f"K-Means clusters (K={best_k})")
km_profile = describe(kmeans.labels_, all_rows)
movies["cluster_label"] = movies.kmeans_cluster.map(km_profile.set_index("cluster").label)
km_profile

**Interpretation:** read each row — `top_genres` and `top_words` tell what the cluster contains (name it yourself, e.g. "Drama / relationships"). The 2-D plot is only a projection, so clusters may overlap visually.

# Section 9 — Model 3: Hierarchical Clustering
| K-Means | Hierarchical |
|---|---|
| Divides data into **K predefined clusters** | Builds a **tree (hierarchy)** of similar points, shown as a **dendrogram** |
| K needed in advance | Cut the tree at any level |
| Scales to all movies | Slow on large data → run on a **random sample** |

**Linkage = Ward:** merges the pair of clusters that increases within-cluster variance the least (same goal as K-Means, so results are comparable).

In [ ]:
n_h = min(1500, len(movies))
h_rows = np.sort(np.random.default_rng(SEED).choice(len(movies), n_h, replace=False))
Xh = X_cluster[h_rows]

L = linkage(Xh, method="ward")
cut = (L[-(best_k - 1), 2] + L[-best_k, 2]) / 2                              # height that gives best_k clusters
plt.figure(figsize=(11, 4.5))
dendrogram(L, truncate_mode="lastp", p=30, leaf_rotation=90, leaf_font_size=9, color_threshold=cut)
plt.axhline(cut, ls="--", c="k"); plt.title(f"Dendrogram (Ward, {n_h} movies)"); plt.ylabel("Distance"); plt.show()

h_labels = fcluster(L, t=best_k, criterion="maxclust") - 1
h_sil, km_sil_same = silhouette_score(Xh, h_labels), silhouette_score(Xh, kmeans.labels_[h_rows])
ari = adjusted_rand_score(kmeans.labels_[h_rows], h_labels)
print(f"Silhouette -> Hierarchical: {h_sil:.4f} | K-Means (same movies): {km_sil_same:.4f} | ARI between them: {ari:.3f}")

plot2d(h_rows, h_labels, f"Hierarchical clusters (Ward, K={best_k})")
describe(h_labels, h_rows)

**Interpretation:** tall branches join very different groups, short branches join near-identical movies; the dashed line is where the tree is cut. ARI shows how similar the K-Means and hierarchical groupings are. Here it is an *analysis* tool, not the recommender.

# Section 10 — Model 4: Random Forest (Preference Prediction)
- **Target:** `like` (rating ≥ 4) — comes from real ratings, nothing fabricated → **binary classification**.
- **Features:** user history, movie history, `vote_average`, `vote_count`, `release_year`, genre flags.
- **No leakage:** split first; history stats use **training rows only** and exclude the row's own rating (leave-one-out).
- **Limitation:** random split, not time-based.

In [ ]:
rf_df = df[["userId", "movieId", "rating", "like"]].merge(
    movies[["movieId", "vote_average", "vote_count", "release_year"] + genre_cols], on="movieId")
rf_df = rf_df.sample(min(len(rf_df), 300_000), random_state=SEED)           # cap for speed on huge data
train, test = train_test_split(rf_df, test_size=0.2, random_state=SEED, stratify=rf_df["like"])

def history(train, test, key, p):
    # mean rating / like-rate / count of a user or movie, from training data only
    g = train[["rating", "like"]].mean()
    s = train.groupby(key).agg(sr=("rating", "sum"), sl=("like", "sum"), n=("rating", "size"))
    t = train[[key, "rating", "like"]].join(s, on=key); n = (t["n"] - 1).replace(0, np.nan)   # leave-one-out
    tr = pd.DataFrame({p + "_cnt": t["n"] - 1, p + "_mean": ((t["sr"] - t["rating"]) / n).fillna(g["rating"]),
                       p + "_like": ((t["sl"] - t["like"]) / n).fillna(g["like"])})
    e = test[[key]].join(s, on=key)
    te = pd.DataFrame({p + "_cnt": e["n"].fillna(0), p + "_mean": (e["sr"] / e["n"]).fillna(g["rating"]),
                       p + "_like": (e["sl"] / e["n"]).fillna(g["like"])})
    return tr, te

(u_tr, u_te), (m_tr, m_te) = history(train, test, "userId", "u"), history(train, test, "movieId", "m")
meta = ["vote_average", "vote_count", "release_year"] + genre_cols
X_tr, X_te = pd.concat([u_tr, m_tr, train[meta]], axis=1), pd.concat([u_te, m_te, test[meta]], axis=1)
y_tr, y_te = train["like"], test["like"]

rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=SEED).fit(X_tr, y_tr)
pred, proba = rf.predict(X_te), rf.predict_proba(X_te)[:, 1]

majority = max(y_te.mean(), 1 - y_te.mean())
rf_metrics = {"Accuracy": accuracy_score(y_te, pred), "Precision": precision_score(y_te, pred),
              "Recall": recall_score(y_te, pred), "F1": f1_score(y_te, pred), "ROC-AUC": roc_auc_score(y_te, proba)}
print(f"Majority-class baseline accuracy: {majority:.4f}")
print({k: round(v, 4) for k, v in rf_metrics.items()})

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ConfusionMatrixDisplay.from_predictions(y_te, pred, display_labels=["Dislike", "Like"], cmap="Blues", ax=ax[0])
pd.Series(rf.feature_importances_, index=X_tr.columns).sort_values().tail(10).plot.barh(ax=ax[1], color="darkgreen")
ax[0].set_title("Confusion matrix (test)"); ax[1].set_title("Top 10 feature importances")
plt.tight_layout(); plt.show()

# Error analysis: is the model worse for users with little history?
print("\nError rate by user-history size (quartiles):")
print((pred != y_te).groupby(pd.qcut(X_te["u_cnt"], 4, duplicates="drop")).mean().round(3))

**Interpretation:** accuracy must be compared with the **majority-class baseline**; precision, recall, F1 and ROC-AUC show how well the *Like* class is predicted. Importance ranking is indicative (impurity-based). The error table shows whether predictions are weaker for users with little history (cold-start).

# Section 11 — Model Comparison
| Model | Type | Purpose | Output | Evaluation |
|---|---|---|---|---|
| **KNN** | Instance-based | Similarity / recommendation | Top-N movies | Precision@K, Recall@K, Hit-rate@K |
| **K-Means** | Unsupervised | Movie segmentation | Clusters | Inertia, Elbow, Silhouette |
| **Hierarchical** | Unsupervised | Similarity / grouping | Dendrogram + clusters | Silhouette, dendrogram |
| **Random Forest** | Supervised ensemble | Preference prediction | Class + P(like) | Accuracy, P, R, F1, AUC, Confusion matrix |

> The models solve **different tasks**, so they are *not* ranked by one accuracy number.

In [ ]:
pd.DataFrame([
    ["KNN", f"Precision@10 = {knn_results.loc[FINAL, 'Precision@10']}, Recall@10 = {knn_results.loc[FINAL, 'Recall@10']}, "
            f"HitRate@10 = {knn_results.loc[FINAL, 'HitRate@10']}"],
    ["K-Means", f"K = {best_k}, silhouette = {km_sil:.4f}, inertia = {kmeans.inertia_:.1f}"],
    ["Hierarchical", f"K = {best_k}, silhouette = {h_sil:.4f}, ARI vs K-Means = {ari:.3f}"],
    ["Random Forest", f"Acc = {rf_metrics['Accuracy']:.3f} (majority = {majority:.3f}), F1 = {rf_metrics['F1']:.3f}, AUC = {rf_metrics['ROC-AUC']:.3f}"],
], columns=["Model", "Actual results from this run"])

# Section 12 — Final Recommendation Logic
**KNN is the final recommendation engine** because it:
1. directly returns a ranked Top-N list (similar movies, or personalised for a user from their liked movies),
2. works for new movies (cold-start) — only genres/overview are needed,
3. is easy to explain (shared genres/words + similarity score),
4. is evaluated with recommendation metrics (Precision@K / Recall@K).

**K-Means & Hierarchical** = supporting analysis (catalogue structure, movie segments). **Random Forest** = preference-prediction experiment.

# Section 13 — Final Recommendation Function
`recommend_movies(title, top_n)` and `recommend_for_user(user_id, top_n)` are saved in `recommender.py` so the **notebook and Streamlit use the same code**. It finds the movie → takes its feature vector → KNN finds nearest neighbours → returns Top-N. Invalid titles get a friendly message with suggestions.

In [ ]:
%%writefile recommender.py
import difflib
import joblib
import numpy as np
import pandas as pd

_cache = {}

def load_bundle(path="models/recommender_bundle.joblib"):
    if path not in _cache:
        _cache[path] = joblib.load(path)
    return _cache[path]

def find_movie(query, bundle=None):
    """Return (row index, suggestions). Exact match first, then partial; most-rated movie wins."""
    movies = (bundle or load_bundle())["movies"]
    q, keys = str(query).lower().strip(), movies["title_key"]
    for hit in (movies[keys == q], movies[keys.str.contains(q, regex=False)] if q else movies.iloc[:0]):
        if len(hit):
            return int(hit["n_ratings"].idxmax()), []
    close = difflib.get_close_matches(q, keys.tolist(), n=5, cutoff=0.5)
    return None, [movies.loc[keys == c, "title"].iloc[0] for c in close]

def recommend_movies(movie_title, top_n=10, bundle=None):
    """Top-N movies most similar to movie_title (KNN, cosine similarity)."""
    b = bundle or load_bundle()
    idx, suggestions = find_movie(movie_title, b)
    if idx is None:
        print(f"Movie '{movie_title}' not found." + (" Did you mean: " + "; ".join(suggestions) if suggestions else ""))
        return pd.DataFrame()
    dist, ind = b["knn_model"].kneighbors(b["movie_matrix"][idx], n_neighbors=int(top_n) + 1)
    keep = ind[0] != idx                                           # remove the movie itself
    dist, ind = dist[0][keep][:top_n], ind[0][keep][:top_n]
    recs = b["movies"].iloc[ind][["title", "genres", "release_year", "vote_average", "n_ratings"]].copy()
    recs["similarity"] = (1 - dist).round(4)
    recs.insert(0, "Rank", np.arange(1, len(recs) + 1))
    return recs.reset_index(drop=True)

def recommend_for_user(user_id, top_n=10, bundle=None):
    """Personalised Top-N: neighbours of the user's liked movies (rating >= 4), already-rated movies hidden."""
    b = bundle or load_bundle()
    seeds = b["liked"].get(user_id, [])[-20:]
    if not seeds:
        print(f"User {user_id} not found or has no liked movies.")
        return pd.DataFrame()
    dist, ind = b["knn_model"].kneighbors(b["movie_matrix"][seeds], n_neighbors=51)
    score = {}
    for d_row, i_row in zip(dist, ind):
        for d, i in zip(d_row, i_row):
            if i not in b["seen"][user_id]:
                score[i] = score.get(i, 0) + 1 - d
    top = sorted(score, key=score.get, reverse=True)[:int(top_n)]
    recs = b["movies"].iloc[top][["title", "genres", "release_year", "vote_average"]].copy()
    recs["score"] = [round(score[i], 3) for i in top]
    recs.insert(0, "Rank", np.arange(1, len(recs) + 1))
    return recs.reset_index(drop=True)

In [ ]:
# Save everything the Streamlit app needs
export_cols = ["movieId", "title", "title_key", "genres", "release_year", "vote_average", "n_ratings", "cluster_label"]
liked_rows = {u: [movie_row[m] for m in g] for u, g in df[df.like == 1].groupby("userId").movieId}   # user -> liked movies
seen_rows = {u: {movie_row[m] for m in g} for u, g in df.groupby("userId").movieId}                  # user -> all rated movies
joblib.dump({"knn_model": knn_model, "movie_matrix": X_content, "movies": movies[export_cols],
             "liked": liked_rows, "seen": seen_rows}, "models/recommender_bundle.joblib", compress=3)
joblib.dump({"model": rf, "features": list(X_tr.columns)}, "models/random_forest.joblib")

from recommender import recommend_movies, recommend_for_user, find_movie
print(recommend_movies("this movie does not exist", 5))          # invalid title is handled gracefully

# Section 14 — Final Results
Sample similar-movie and personalised recommendations. If a title is not in this dataset, the function says so (nothing is invented); the two most-rated movies are added so there is always output.

In [ ]:
for title in ["Interstellar", "Inception", "The Dark Knight", "Avengers"] + movies.nlargest(2, "n_ratings").title.tolist():
    idx, _ = find_movie(title)
    if idx is None:
        recommend_movies(title, 5); continue
    print(f"\n=== {title} -> {movies.title[idx]} | segment: {movies.cluster_label[idx]} ===")
    print(recommend_movies(title, 5).to_string(index=False))

# Personalised recommendation for the user with most liked movies
u = max(liked_rows, key=lambda k: len(liked_rows[k]))
print(f"\n=== Personalised: user {u} (liked {len(liked_rows[u])} movies, e.g. {movies.title[liked_rows[u][:3]].tolist()}) ===")
print(recommend_for_user(u, 5).to_string(index=False))

### Streamlit app
Run: `streamlit run app.py` (needs `models/` and `recommender.py` in the same folder).

In [ ]:
%%writefile app.py
import streamlit as st
from recommender import load_bundle, find_movie, recommend_movies, recommend_for_user

st.set_page_config(page_title="Movie Recommender", layout="wide")
st.title("🎬 Movie Recommendation System")
st.caption("KNN on TF-IDF content features · personalised from users' liked movies")

bundle = st.cache_resource(load_bundle)()
movies = bundle["movies"]

tab1, tab2 = st.tabs(["Similar movies", "Personalised for a user"])

with tab1:
    choice = st.selectbox("Search or select a movie", sorted(movies["title"].unique()))
    top_n = st.slider("Number of recommendations", 5, 20, 10)
    if st.button("Recommend"):
        info = movies.loc[find_movie(choice, bundle)[0]]
        st.write(f"**{info['title']}** · {info['genres']} · segment: *{info['cluster_label']}*")
        st.dataframe(recommend_movies(choice, top_n, bundle), hide_index=True)

with tab2:
    user_id = st.selectbox("User ID", sorted(bundle["liked"]))
    n_user = st.slider("Number of recommendations", 5, 20, 10, key="n_user")
    if st.button("Recommend for user"):
        st.caption(f"Based on {len(bundle['liked'][user_id])} movies this user liked (rating ≥ 4)")
        st.dataframe(recommend_for_user(user_id, n_user, bundle), hide_index=True)

# Section 15 — Conclusion
- **Achieved:** complete pipeline — cleaning, EDA, features, 4 models, saved artifacts, Streamlit app.
- **Best for recommendation:** **KNN** (only model that outputs a ranked list, also personalised from a user's liked movies); see the Precision@K table for KNN variants vs baselines.
- **Clustering:** K-Means gives flat movie segments; Hierarchical shows how groups merge (dendrogram). Both are supporting analysis.
- **Random Forest:** predicts like/dislike from real ratings, checked against a majority baseline.
- **Limitations:** content-only KNN ignores who rated what · only observed ratings used for evaluation · random (not time-based) split · hierarchical clustering run on a sample.
- **Future work:** collaborative filtering / hybrid model · sentence-embedding text features · cast & director features · time-based evaluation.

---
**Removed models:** Logistic Regression, Decision Tree, Naive Bayes, SVM, Gradient Boosting
**Final models:** KNN → recommendation · K-Means → segmentation · Hierarchical → similarity/grouping · Random Forest → preference prediction
**Architecture:** Dataset → Cleaning → EDA → Feature Engineering → KNN → K-Means → Hierarchical → Random Forest → Analysis → Final KNN Recommender → Streamlit